# 14b: Statistical Analysis of Clinical Imaging Features vs Rejection

Source data: `../data/bd_estudiUPF.csv` (original clinical spreadsheet)

This notebook analyzes the ARFI elastography and DCE-US (contrast-enhanced ultrasound)
parameters that were manually measured by the radiologists. These are the same features
that Clara's paper (Bassaganyas et al. 2025) analyzed.

Features:
- ARFI: mediana, media, DE, RIQ (elastography/stiffness)
- DCE-US: PE, WiAUC, RT, mTTI, TTP, WiR, WiPi, WoAUC, WiWoAUC, FT, WoR (perfusion)
- QOF, Area (quality and region size)

Goal: reproduce Clara's analysis and compare with our radiomics results.

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multitest import multipletests
import os

print("Imports OK")

In [ ]:
# load clinical data directly from source
clinical = pd.read_csv(os.path.join("..", "data", "bd_estudiUPF.csv"))
clinical["id estudio"] = clinical["id estudio"].astype(str).str.strip()
print(f"Clinical data: {clinical.shape[0]} rows, {clinical.shape[1]} columns")

# rename key columns for convenience
motivo_col = "motivo (1: 1 semana, 2: 1 mes, 3: 1 año, 4: sospecha, 5: seguimiento)="
clinical = clinical.rename(columns={
    "id estudio": "study_id",
    motivo_col: "motivo",
    "RECHAZO CLÍNICO": "rejection",
})

print(f"Rejection distribution:")
print(clinical["rejection"].value_counts().to_string())

In [ ]:
# define the clinical imaging features to test
# ARFI elastography (stiffness)
arfi_features = ["ARFI mediana", "ARFI media", "ARFI DE", "ARFI RIQ"]

# DCE-US perfusion parameters
dceus_features = ["PE", "WiAUC", "RT", "mTTI", "TTP", "WiR", "WiPi",
                  "WoAUC", "WiWoAUC", "FT", "WoR"]

# other
other_features = ["QOF", "Area"]

all_clinical_features = arfi_features + dceus_features + other_features

# check availability and non-null counts
print("Feature availability:")
for feat in all_clinical_features:
    if feat in clinical.columns:
        n_valid = clinical[feat].dropna().shape[0]
        print(f"  {feat}: {n_valid}/{len(clinical)} non-null")
    else:
        print(f"  {feat}: NOT FOUND")

In [ ]:
# split into groups
rejection = clinical[clinical["rejection"] == 1]
no_rejection = clinical[clinical["rejection"] == 0]

print(f"No rejection: {len(no_rejection)}")
print(f"Rejection: {len(rejection)}")

## Univariate testing -- full dataset
Using Mann-Whitney U (consistent with Clara's paper).

In [ ]:
test_results = []

for feat in all_clinical_features:
    if feat not in clinical.columns:
        continue

    vals_no_rej = no_rejection[feat].dropna().values
    vals_rej = rejection[feat].dropna().values

    if len(vals_no_rej) < 3 or len(vals_rej) < 3:
        print(f"  Skipping {feat}: too few values")
        continue

    stat, p_value = stats.mannwhitneyu(vals_no_rej, vals_rej, alternative="two-sided")

    # rank-biserial correlation as effect size
    n1 = len(vals_no_rej)
    n2 = len(vals_rej)
    effect_size = 1 - (2 * stat) / (n1 * n2)

    test_results.append({
        "feature": feat,
        "n_no_rej": n1,
        "n_rej": n2,
        "statistic": stat,
        "p_value": p_value,
        "effect_size": effect_size,
        "median_no_rej": np.median(vals_no_rej),
        "median_rej": np.median(vals_rej),
    })

results_df = pd.DataFrame(test_results).sort_values("p_value").reset_index(drop=True)
print(f"Tests run: {len(results_df)}")

In [ ]:
# show all results
significant = results_df[results_df["p_value"] < 0.05]
print(f"Features with p < 0.05: {len(significant)} out of {len(results_df)}")
print()

display_cols = ["feature", "p_value", "effect_size", "median_no_rej", "median_rej", "n_no_rej", "n_rej"]
print("All features (sorted by p-value):")
print(results_df[display_cols].to_string(index=False))

## Stratified by time period
Clara found that ARFI and DCE-US parameters are only discriminative after 90 days post-transplant.

In [ ]:
def run_mannwhitney_on_subset(subset_df, label):
    """Run Mann-Whitney on clinical features for a subset."""
    rej = subset_df[subset_df["rejection"] == 1]
    no_rej = subset_df[subset_df["rejection"] == 0]

    print(f"\n--- {label} ---")
    print(f"No rejection: {len(no_rej)}, Rejection: {len(rej)}")

    if len(rej) < 3 or len(no_rej) < 3:
        print("Too few samples in one group. Skipping.")
        return None

    rows = []
    for feat in all_clinical_features:
        if feat not in subset_df.columns:
            continue
        vals_no = no_rej[feat].dropna().values
        vals_yes = rej[feat].dropna().values
        if len(vals_no) < 3 or len(vals_yes) < 3:
            continue
        stat, p = stats.mannwhitneyu(vals_no, vals_yes, alternative="two-sided")
        effect = 1 - (2 * stat) / (len(vals_no) * len(vals_yes))
        rows.append({
            "feature": feat,
            "p_value": p,
            "effect_size": effect,
            "median_no_rej": np.median(vals_no),
            "median_rej": np.median(vals_yes),
        })

    sub_results = pd.DataFrame(rows).sort_values("p_value")
    sig = sub_results[sub_results["p_value"] < 0.05]
    print(f"Features with p < 0.05: {len(sig)}")
    print()
    print(sub_results[["feature", "p_value", "effect_size", "median_no_rej", "median_rej"]].to_string(index=False))

    return sub_results

early_results = run_mannwhitney_on_subset(
    clinical[clinical["motivo"].isin([1, 2])],
    "Early (motivo 1-2, within 90 days)")

late_results = run_mannwhitney_on_subset(
    clinical[clinical["motivo"].isin([3, 4, 5])],
    "Late (motivo 3-5, beyond 90 days)")

In [ ]:
# save
output_path = os.path.join("reports", "stats_clinical_features.csv")
results_df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

print()
print("SUMMARY")
print(f"Total clinical features tested: {len(results_df)}")
print(f"Uncorrected p < 0.05: {len(results_df[results_df['p_value'] < 0.05])}")